# 📋 데이터 점검 코드
V10 제작 전 데이터 상태 확인용

## STEP 0: 환경 설정 및 경로 확인

In [ ]:
import os
import pandas as pd
import numpy as np

# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# 경로 설정
base_path = '/content/drive/MyDrive/AI_Projects/smoking_hackathon/'
data_path = base_path + 'data/'

train_path = data_path + 'train.csv'
test_path = data_path + 'test.csv'
submission_path = data_path + 'sample_submission.csv'

## STEP 1: 파일 존재 여부 확인

In [ ]:
print("="*60)
print("📁 파일 존재 여부 확인")
print("="*60)

files_to_check = {
    'train.csv': train_path,
    'test.csv': test_path,
    'sample_submission.csv': submission_path
}

file_status = []
for name, path in files_to_check.items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    file_status.append({
        '파일명': name,
        '경로': path,
        '존재': '✅' if exists else '❌',
        '크기(KB)': round(size/1024, 2) if exists else '-'
    })

status_df = pd.DataFrame(file_status)
print(status_df.to_markdown(index=False))

## STEP 2: 데이터 Shape 및 컬럼명

In [ ]:
# 데이터 로드
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print("="*60)
print("📊 데이터 Shape")
print("="*60)

shape_info = [
    {'데이터셋': 'train', '행(Rows)': train.shape[0], '열(Columns)': train.shape[1]},
    {'데이터셋': 'test', '행(Rows)': test.shape[0], '열(Columns)': test.shape[1]},
    {'데이터셋': 'submission', '행(Rows)': submission.shape[0], '열(Columns)': submission.shape[1]}
]
print(pd.DataFrame(shape_info).to_markdown(index=False))

In [ ]:
print("\n" + "="*60)
print("📋 Train 컬럼명 (순서대로)")
print("="*60)

col_info = []
for i, col in enumerate(train.columns):
    col_info.append({
        '순번': i+1,
        '컬럼명': col,
        'dtype': str(train[col].dtype),
        'test포함': '✅' if col in test.columns else '❌'
    })

print(pd.DataFrame(col_info).to_markdown(index=False))

In [ ]:
print("\n" + "="*60)
print("🔍 중복 컬럼 확인")
print("="*60)

train_dup = train.columns[train.columns.duplicated()].tolist()
test_dup = test.columns[test.columns.duplicated()].tolist()

print(f"Train 중복 컬럼: {train_dup if train_dup else '없음'}") 
print(f"Test 중복 컬럼: {test_dup if test_dup else '없음'}")

# Train에만 있는 컬럼 (label 제외)
train_only = [c for c in train.columns if c not in test.columns]
print(f"\nTrain에만 있는 컬럼: {train_only}")

## STEP 3: 타깃 컬럼 확인 및 분포

In [ ]:
print("="*60)
print("🎯 타깃 컬럼 확인")
print("="*60)

# 타깃 컬럼 찾기 (label 또는 마지막 컬럼)
target_candidates = [c for c in train.columns if 'label' in c.lower() or 'target' in c.lower()]
print(f"타깃 후보: {target_candidates}")

# label 컬럼이 있다고 가정
target_col = None
for col in train.columns:
    if 'label' in col.lower():
        target_col = col
        break

if target_col is None:
    target_col = train.columns[-1]
    print(f"⚠️ label 컬럼 없음, 마지막 컬럼 사용: {target_col}")
else:
    print(f"✅ 타깃 컬럼: {target_col}")

# 분포
print(f"\n📊 타깃 분포:")
dist = train[target_col].value_counts().sort_index()
total = len(train)

dist_info = []
for val, cnt in dist.items():
    dist_info.append({
        '값': val,
        '개수': cnt,
        '비율(%)': round(cnt/total*100, 2)
    })
print(pd.DataFrame(dist_info).to_markdown(index=False))

# 클래스 불균형 비율
if len(dist) == 2:
    imbalance = dist.max() / dist.min()
    print(f"\n불균형 비율: {imbalance:.2f}:1")

## STEP 4: 결측치 비율 요약

In [ ]:
print("="*60)
print("❓ 결측치 비율")
print("="*60)

def missing_summary(df, name):
    missing = df.isnull().sum()
    pct = (missing / len(df) * 100).round(2)
    summary = pd.DataFrame({
        '컬럼': df.columns,
        '결측수': missing.values,
        '결측률(%)': pct.values
    })
    summary = summary[summary['결측수'] > 0]
    if len(summary) == 0:
        print(f"\n{name}: 결측치 없음 ✅")
    else:
        print(f"\n{name}:")
        print(summary.to_markdown(index=False))
    return summary

train_missing = missing_summary(train, 'Train')
test_missing = missing_summary(test, 'Test')

print(f"\n📊 전체 요약:")
print(f"   Train 총 결측: {train.isnull().sum().sum()}")
print(f"   Test 총 결측: {test.isnull().sum().sum()}")

## STEP 5: 수치 컬럼 기초 통계

In [ ]:
print("="*60)
print("📈 수치 컬럼 기초 통계")
print("="*60)

numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
if target_col in numeric_cols:
    numeric_cols.remove(target_col)

print(f"\n수치 컬럼 ({len(numeric_cols)}개): {numeric_cols}")

# describe
stats = train[numeric_cols].describe().T
stats = stats.round(2)
print("\n" + stats.to_markdown())

In [ ]:
# 분위수 상세
print("\n" + "="*60)
print("📊 분위수 상세 (1%, 5%, 25%, 50%, 75%, 95%, 99%)")
print("="*60)

quantiles = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
q_stats = train[numeric_cols].quantile(quantiles).T
q_stats.columns = [f'{int(q*100)}%' for q in quantiles]
print("\n" + q_stats.round(2).to_markdown())

In [ ]:
# 범주형 컬럼 확인
print("\n" + "="*60)
print("📋 범주형 후보 (고유값 ≤ 10개)")
print("="*60)

cat_candidates = []
for col in numeric_cols:
    nunique = train[col].nunique()
    if nunique <= 10:
        cat_candidates.append({
            '컬럼': col,
            '고유값수': nunique,
            '값': sorted(train[col].unique().tolist())
        })

if cat_candidates:
    print(pd.DataFrame(cat_candidates).to_markdown(index=False))
else:
    print("범주형 후보 없음")

## STEP 6: 요약 테이블 출력

In [ ]:
print("\n" + "="*60)
print("📋 최종 요약")
print("="*60)

summary_table = {
    '항목': [
        'Train 크기',
        'Test 크기',
        '피처 수 (타깃 제외)',
        '타깃 컬럼',
        '타깃 클래스 수',
        '클래스 0 비율',
        '클래스 1 비율',
        '총 결측치',
        '수치 컬럼 수',
        '범주형 후보 수'
    ],
    '값': [
        f"{train.shape[0]:,} rows × {train.shape[1]} cols",
        f"{test.shape[0]:,} rows × {test.shape[1]} cols",
        len(train.columns) - 1,
        target_col,
        train[target_col].nunique(),
        f"{(train[target_col]==0).sum():,} ({(train[target_col]==0).mean()*100:.1f}%)",
        f"{(train[target_col]==1).sum():,} ({(train[target_col]==1).mean()*100:.1f}%)",
        train.isnull().sum().sum() + test.isnull().sum().sum(),
        len(numeric_cols),
        len(cat_candidates)
    ]
}

print(pd.DataFrame(summary_table).to_markdown(index=False))

print("\n✅ 데이터 점검 완료!")